# 11-加餐：补齐基础拼图（Functional API / Send / 缓存 / 递归限制）

## 把官方文档里“常用但容易漏”的能力补齐

Functional API · Send/并行 · Node caching · recursion_limit · Overwrite · Pydantic · 第十一课 · 约 50 分钟

### 目录

- 回顾：01–10 主线已经够用，但官方还有一套“工具箱”
- 一、两套写法：Graph API vs Functional API
- 二、Functional API：@entrypoint + @task + interrupt/resume
- 三、Send：把并行/Map-Reduce 写成一行路由
- 四、Node caching：cache + CachePolicy（让重复计算变快）
- 五、recursion_limit：循环的保险丝 + RemainingSteps（优雅收尾）
- 六、Overwrite：绕过 reducer 的“强制覆盖”
- 七、Private state + Pydantic：更严格的 schema 与更干净的数据流
- 八、动手跑一下
- 九、检查理解
- 十、常见坑速查
- 十一、总结
- 📖 参考（官方）


### 回顾：01–10 主线已经够用，但官方还有一套“工具箱”

你已经学完了 Graph API 的主干能力（控制流 / 持久化 / HIL / streaming / store / subgraph / 容错 / 工程化）。

但官方文档里还有一些**基础功能**，非常常用，却很容易在“教程主线”里漏掉：

- **Functional API**：不用显式画图，也能用 LangGraph 的持久化/中断/streaming
- **Send**：把“并行/Map-Reduce”写成一行（而不是手写线程/队列）
- **Node caching**：重复计算直接命中缓存
- **recursion_limit / RemainingSteps**：循环与 agent loop 的保险丝
- **Overwrite / Pydantic / Private state**：让 state 既严格又好维护

这一节把它们补齐，你读官方文档就不会再“缺块拼图”。

In [1]:
import time
import uuid
import operator
from typing import Annotated, Literal

from typing_extensions import TypedDict


def now_ms() -> int:
    return int(time.time() * 1000)


def print_title(title: str):
    print("\n" + "=" * 20)
    print(title)
    print("=" * 20)

---

### 一、两套写法：Graph API vs Functional API

你可以把它们理解成：

- **Graph API**：先“画骨架”（节点/边/状态），再让它跑。
- **Functional API**：先“写业务代码”（if/for/函数调用），再把持久化/中断/streaming“嵌进去”。

选择原则（工程上最实用）：

- 需要**可视化、路由结构清晰、模块化（subgraph）** → Graph API
- 你已经有一堆现成业务函数，想最小改动加上**持久化/中断/并行 task** → Functional API

这两套 API 底层共享同一个运行时，所以**可以混用**。

---

### 二、Functional API：@entrypoint + @task + interrupt/resume

Functional API 的两块积木：

- `@entrypoint`：把一个函数变成“可持久化、可中断、可 streaming”的工作流
- `@task`：把“可能慢/可能有副作用”的步骤包成任务

关键直觉（很像我们第 05 节讲的 interrupt）：

- **恢复执行时，会从头重新跑 entrypoint**
- 但 **已经完成的 task 结果会从 checkpoint 恢复**（避免重复计算/重复调用）

所以：

- 非确定性/副作用尽量放进 task（或者保证幂等）
- interrupt 的顺序要稳定（否则 resume 值会对不上）


In [2]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.func import entrypoint, task
from langgraph.types import Command, interrupt


@task
def write_draft(topic: str) -> str:
    # 模拟慢操作（真实场景可能是 LLM / 外部 API）
    time.sleep(0.2)
    return f"《{topic}》初稿：...（离线示例）"


@entrypoint(checkpointer=InMemorySaver())
def workflow(topic: str) -> dict:
    draft = write_draft(topic).result()
    approved = interrupt({
        "draft": draft,
        "action": "请审批：approve / reject",
    })
    return {
        "draft": draft,
        "approved": approved,
    }


thread_id = f"fn-{uuid.uuid4().hex[:8]}"
config_fn = {"configurable": {"thread_id": thread_id}}

"ready"

'ready'

> **❓ 检查理解 ①**
>
> 在 Functional API + checkpointer + interrupt 的场景里，哪种写法最能避免“恢复后重复副作用/重复耗时”？
>
> - A. 把慢操作写在 entrypoint 里（`time.sleep()` / HTTP / LLM），然后再 `interrupt()`
> - B. 把慢操作写成 `@task`，entrypoint 里只 `task(...).result()`，再 `interrupt()`
> - C. 完全不需要考虑，resume 一定会从 interrupt 之后继续跑，不会重跑前面的代码
>
> **✅ 答案：B**
>
> 解释：恢复时 entrypoint 会从头执行，但已完成的 task 结果会从 checkpoint 恢复；A 会让恢复时重复耗时/重复副作用；C 是误解（“从头重跑”是核心语义）。


---

### 三、Send：把并行/Map-Reduce 写成一行路由

`Send` 的用途是：**一次路由，派发 N 个“同类型任务节点”并行执行**。

典型形态就是 Map-Reduce：

- Map：对每个 item 做一次处理（并行）
- Reduce：把结果合并回 state（靠 reducer）

下面我们写一个离线小例子：给多个商品做“规则检查”，把检查结果收集到 `reports` 列表里。

In [3]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send


class SendState(TypedDict):
    items: list[str]
    reports: Annotated[list[str], operator.add]


def plan_items(state: SendState) -> list[Send]:
    # 一次性派发多个并行节点
    return [Send("检查", {"item": x}) for x in state["items"]]


def check_one(state: dict) -> dict:
    item = state["item"]
    ok = "坏" not in item
    return {"reports": [f"{item}: {'OK' if ok else 'BLOCK'}"]}


def summarize(state: SendState) -> dict:
    return {"reports": [f"总计 {len(state['reports'])} 条检查报告"]}


builder_send = StateGraph(SendState)
builder_send.add_node("检查", check_one)
builder_send.add_node("汇总", summarize)

builder_send.add_conditional_edges(START, plan_items)
builder_send.add_edge("检查", "汇总")
builder_send.add_edge("汇总", END)

graph_send = builder_send.compile()

init_send: SendState = {"items": ["商品A", "坏商品B", "商品C"], "reports": []}
"ready"


'ready'

> **❓ 检查理解 ②**
>
> `Send("检查", {"item": x})` 的效果最接近下面哪一种描述？
>
> - A. 立即执行节点“检查”，并把返回值直接合并进 state
> - B. 向运行时“派发一个待执行的节点调用”，可能与其它 Send 并行执行
> - C. 把 x 追加到 state 里，等下一轮 `invoke()` 再执行
>
> **✅ 答案：B**
>
> 解释：`Send` 是“派发/调度”，不是立即执行；真正执行发生在该 superstep 的调度阶段，合并结果靠 reducer。


---

### 四、Node caching：cache + CachePolicy（让重复计算变快）

有些节点很贵：

- 同一个输入会被反复计算（比如：同一个订单规则检查、同一个 prompt 生成）
- 你希望“第二次直接命中缓存”

LangGraph 的节点缓存需要两件事：

- compile 时提供 `cache=`（例如 `InMemoryCache()`）
- node 上配置 `cache_policy=CachePolicy(ttl=...)`

下面我们用一个 `sleep` 模拟昂贵计算，并观察第二次调用会出现 `__metadata__:{cached: True}`。

In [4]:
from langgraph.cache.memory import InMemoryCache
from langgraph.types import CachePolicy


class CacheState(TypedDict):
    x: int
    result: int


def expensive_node(state: CacheState) -> dict:
    time.sleep(0.2)
    return {"result": state["x"] * 2}


builder_cache = StateGraph(CacheState)
builder_cache.add_node("expensive", expensive_node, cache_policy=CachePolicy(ttl=3))
builder_cache.add_edge(START, "expensive")
builder_cache.add_edge("expensive", END)

graph_cache = builder_cache.compile(cache=InMemoryCache())

"ready"

'ready'

---

### 五、recursion_limit：循环的保险丝 + RemainingSteps（优雅收尾）

当你写 agent loop / 自循环时，最怕两件事：

- 没有退出条件 → 无限循环
- 退出条件写错 → 跑到 1000 步才发现（浪费时间/钱）

官方提供了两层保护：

- `recursion_limit`：运行时上限（超过就抛 `GraphRecursionError`）
- `RemainingSteps`：把“剩余步数”作为 managed value 注入到 state，让你在图内优雅收尾

下面我们做一个最小 loop：正常会循环，但在 `RemainingSteps` 快耗尽时自动走到 END。

In [5]:
from langgraph.errors import GraphRecursionError
from langgraph.managed import RemainingSteps


class LoopState(TypedDict):
    steps: Annotated[list[str], operator.add]
    remaining_steps: RemainingSteps


def loop_node(state: LoopState) -> dict:
    remaining = state["remaining_steps"]
    if remaining <= 2:
        return {"steps": [f"剩余步数={remaining}，准备收尾"]}
    return {"steps": [f"继续循环（剩余步数={remaining}）"]}


def route_loop(state: LoopState) -> Literal["loop", END]:
    if state["remaining_steps"] <= 2:
        return END
    return "loop"


builder_loop = StateGraph(LoopState)
builder_loop.add_node("loop", loop_node)
builder_loop.add_edge(START, "loop")
builder_loop.add_conditional_edges("loop", route_loop)

graph_loop = builder_loop.compile()

# 对照：没有 RemainingSteps 的“无脑循环”（用来演示 GraphRecursionError）
class BadLoopState(TypedDict):
    steps: Annotated[list[str], operator.add]


def bad_loop(state: BadLoopState) -> dict:
    return {"steps": ["一直循环..."]}


def bad_route(_: BadLoopState) -> Literal["bad", END]:
    return "bad"


builder_bad = StateGraph(BadLoopState)
builder_bad.add_node("bad", bad_loop)
builder_bad.add_edge(START, "bad")
builder_bad.add_conditional_edges("bad", bad_route)
graph_bad = builder_bad.compile()

"ready"

'ready'

> **❓ 检查理解 ③**
>
> `recursion_limit` 应该放在 config 的哪里？
>
> - A. `config={"configurable": {"recursion_limit": 5}}`
> - B. `config={"recursion_limit": 5}`
> - C. `context={"recursion_limit": 5}`
>
> **✅ 答案：B**
>
> 解释：`recursion_limit` 是一个独立的顶层 config key，不放在 `configurable` 里。


---

### 六、Overwrite：绕过 reducer 的“强制覆盖”

当一个 key 配了 reducer（比如 `operator.add`），节点返回的新值默认会被“合并”。

但有时候你就是想**强制覆盖**（例如：把对话历史替换成一个“修正版”）。

这时用 `Overwrite(value)`：它会绕过 reducer，直接把 channel 设置成你给的值。

下面我们用一个 list reducer 演示：同样是更新 `messages`，默认会 append，但 Overwrite 会直接替换。

In [6]:
from langgraph.types import Overwrite


class OverwriteState(TypedDict):
    messages: Annotated[list[str], operator.add]


def append_msg(_: OverwriteState) -> dict:
    return {"messages": ["append"]}


def replace_msg(_: OverwriteState) -> dict:
    return {"messages": Overwrite(["replacement"])}


builder_over = StateGraph(OverwriteState)
builder_over.add_node("append", append_msg)
builder_over.add_node("replace", replace_msg)
builder_over.add_edge(START, "append")
builder_over.add_edge("append", "replace")
builder_over.add_edge("replace", END)

graph_over = builder_over.compile()

"ready"

'ready'

---

### 七、Private state + Pydantic：更严格的 schema 与更干净的数据流

两个很实用的“基础工具”：

- **Private state（节点间私有数据）**：只让 node_1 → node_2 共享某个中间字段，避免把一堆临时字段塞进整体 state。
- **Pydantic state schema**：给输入加运行时校验（坏输入尽早失败）。

下面各给一个最小可跑例子。

In [7]:
# Private state: node_1 -> node_2 共享 private_data，node_3 看不到

class OverallState(TypedDict):
    a: str

class Node1Output(TypedDict):
    private_data: str

class Node2Input(TypedDict):
    private_data: str


def node_1(state: OverallState) -> Node1Output:
    out = {"private_data": "set by node_1"}
    print(f"Entered node_1: input={state} returned={out}")
    return out


def node_2(state: Node2Input) -> OverallState:
    out = {"a": f"node_2 saw: {state['private_data']}"}
    print(f"Entered node_2: input={state} returned={out}")
    return out


def node_3(state: OverallState) -> OverallState:
    out = {"a": f"node_3 saw public a={state['a']}"}
    print(f"Entered node_3: input={state} returned={out}")
    return out


builder_private = StateGraph(OverallState).add_sequence([node_1, node_2, node_3])
builder_private.add_edge(START, "node_1")
graph_private = builder_private.compile()

"ready"

'ready'

In [8]:
# Pydantic state schema：输入校验（坏输入尽早失败）

from pydantic import BaseModel, ValidationError


class PState(BaseModel):
    a: str


def p_node(state: PState) -> dict:
    return {"a": "goodbye"}


builder_p = StateGraph(PState)
builder_p.add_node("p_node", p_node)
builder_p.add_edge(START, "p_node")
builder_p.add_edge("p_node", END)
graph_pyd = builder_p.compile()

print("valid:", graph_pyd.invoke({"a": "hello"}))

try:
    graph_pyd.invoke({"a": 123})
except ValidationError as e:
    print("invalid input caught:", type(e).__name__)
    print(str(e).splitlines()[0])

valid: {'a': 'goodbye'}
invalid input caught: ValidationError
1 validation error for PState


---

### 八、动手跑一下

下面把本节所有“工具箱能力”跑一遍，观察输出：

- Functional API：interrupt → resume（用 stream_events v3）
- Send：并行派发
- Node caching：第二次命中 `cached=True`
- RemainingSteps：接近 recursion_limit 时优雅收尾
- Overwrite：绕过 reducer 强制覆盖
- Private state：private_data 只在 node_1→node_2 之间流转


In [9]:
print_title("Functional API: interrupt -> resume")
stream = workflow.stream_events("退款申请说明", config_fn, version="v3")
_ = stream.output
print("interrupted:", stream.interrupted)
print("interrupt payload:", stream.interrupts[0].value)

resumed = workflow.stream_events(Command(resume="approve"), config_fn, version="v3")
print("final output:", resumed.output)

print_title("Send: parallel dispatch")
out_send = graph_send.invoke(init_send)
print(out_send)

print_title("Node caching: second call is cached")
print(graph_cache.invoke({"x": 5}, stream_mode="updates"))
print(graph_cache.invoke({"x": 5}, stream_mode="updates"))

print_title("RemainingSteps: graceful loop")
out_loop = graph_loop.invoke({"steps": []}, config={"recursion_limit": 6})
print(out_loop["steps"])

print_title("Bad loop: GraphRecursionError (caught)")
try:
    graph_bad.invoke({"steps": []}, config={"recursion_limit": 6})
except GraphRecursionError as e:
    print("caught:", type(e).__name__)

print_title("Overwrite: replace bypasses reducer")
out_over = graph_over.invoke({"messages": []})
print(out_over)

print_title("Private state")
out_private = graph_private.invoke({"a": "set at start"})
print("final:", out_private)


Functional API: interrupt -> resume


/home/miao/anaconda3/envs/tp-rag/lib/python3.12/site-packages/langgraph/pregel/main.py:3723: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
/home/miao/anaconda3/envs/tp-rag/lib/python3.12/site-packages/langgraph/pregel/main.py:3573: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)


interrupted: True
interrupt payload: {'draft': '《退款申请说明》初稿：...（离线示例）', 'action': '请审批：approve / reject'}
final output: {'draft': '《退款申请说明》初稿：...（离线示例）', 'approved': 'approve'}

Send: parallel dispatch
{'items': ['商品A', '坏商品B', '商品C'], 'reports': ['商品A: OK', '坏商品B: BLOCK', '商品C: OK', '总计 3 条检查报告']}

Node caching: second call is cached
[{'expensive': {'result': 10}}]
[{'expensive': {'result': 10}, '__metadata__': {'cached': True}}]

RemainingSteps: graceful loop
['继续循环（剩余步数=5）', '继续循环（剩余步数=4）', '继续循环（剩余步数=3）', '剩余步数=2，准备收尾']

Bad loop: GraphRecursionError (caught)
caught: GraphRecursionError

Overwrite: replace bypasses reducer
{'messages': ['replacement']}

Private state
Entered node_1: input={'a': 'set at start'} returned={'private_data': 'set by node_1'}
Entered node_2: input={'private_data': 'set by node_1'} returned={'a': 'node_2 saw: set by node_1'}
Entered node_3: input={'a': 'node_2 saw: set by node_1'} returned={'a': 'node_3 saw public a=node_2 saw: set by node_1'}
final: {'a': 

---

### 九、检查理解

> **❓ 检查理解 ④**
>
> 关于 `Overwrite(value)`，下面哪个说法是正确的？
>
> - A. Overwrite 只对没有 reducer 的 key 生效
> - B. Overwrite 会绕过 reducer，直接把该 key 设置为 value
> - C. Overwrite 会把 value 追加到 reducer 的结果后面
>
> **✅ 答案：B**
>
> 解释：Overwrite 的语义就是“强制覆盖”，用于你不想走 reducer 合并逻辑的场景。

> **❓ 检查理解 ⑤**
>
> 下面哪一项最能解释 Private state 的价值？
>
> - A. 让 node_1 的输出自动写进整体 state，方便 node_3 也能看到
> - B. 让某些中间字段只在特定节点之间流转，避免污染整体 state schema
> - C. 让所有节点都能访问任意临时变量，不需要定义 schema
>
> **✅ 答案：B**
>
> 解释：Private state 的目的就是“中间字段不进入整体 schema”，让整体 state 更干净、更稳定。


---

### 十、常见坑速查

| 症状 | 原因 | 解法 |
| --- | --- | --- |
| Functional API resume 后“又跑了一遍” | entrypoint 恢复语义是从头执行 | 把慢/副作用放进 `@task` 或保证幂等 |
| `Send` 的结果合并乱/丢 | 没给列表字段配 reducer | 并行收集结果必须用 reducer（如 `operator.add`） |
| 缓存不生效 | 只写了 `cache_policy`，没在 compile 传 `cache=` | 两者缺一不可：`compile(cache=...)` + `cache_policy=...` |
| recursion_limit 设置了但没作用 | 放进了 `configurable` | `config={"recursion_limit": N}`（顶层 key） |
| Overwrite 并行时报 InvalidUpdateError | 同一 superstep 多个节点 Overwrite 同一 key | 设计成“单点覆盖”，或拆到不同 superstep |

### 十一、总结

| 工具 | 一句话 | 典型用途 |
| --- | --- | --- |
| Functional API | 让普通函数也拥有持久化/中断/streaming | 最小改动改造老代码 |
| Send | 一次路由派发 N 个并行节点 | Map-Reduce、批量处理 |
| Node caching | 相同输入直接复用结果 | 贵节点去重、节省时间/成本 |
| recursion_limit / RemainingSteps | 循环保险丝 + 优雅收尾 | agent loop、防止无限循环 |
| Overwrite | 绕过 reducer 强制覆盖 | 修正历史、替换列表 |
| Private state | 中间字段不污染整体 schema | 工程整洁、减少 state 膨胀 |
| Pydantic schema | 输入运行时校验 | 早失败、少踩坑 |

📖 参考（官方）

- [Functional API](https://docs.langchain.com/oss/python/langgraph/functional-api)
- [Use Functional API](https://docs.langchain.com/oss/python/langgraph/use-functional-api)
- [Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api)
- [Use Graph API](https://docs.langchain.com/oss/python/langgraph/use-graph-api)
